# Hotel Review Intelligence - API Validation Notebook
Run FastAPI first: `uvicorn backend.api.main:app --reload`

In [1]:
import requests, json, pandas as pd
BASE_URL="http://localhost:8000/api/v1"


In [2]:
health=requests.get(f"{BASE_URL}/health")
print(health.status_code)
print(health.json())

200
{'status': 'healthy'}


In [3]:
models=requests.get(f"{BASE_URL}/models")
print(json.dumps(models.json(),indent=2))

{
  "sentiment_model": "LogisticRegression",
  "rating_model": "LogisticRegression",
  "emotion_model": "j-hartmann/emotion-english-distilroberta-base"
}


## Single Review Test

In [4]:
review = "The historic charm and courtyard gardens created such a romantic atmosphere for our anniversary weekend. The staff even left a complimentary bottle of wine in our room. Unfortunately, the bathroom was severely outdated with a leaky faucet and stained tiles. The clawfoot tub looked nice, but the water pressure was practically non-existent. Lovely ambiance and sweet gestures, but the plumbing definitely needs a modern upgrade."
payload = {"review_text": review}
resp = requests.post(f"{BASE_URL}/analyze", json=payload)
data = resp.json()
print(json.dumps(data, indent=2))

{
  "review_text": "The historic charm and courtyard gardens created such a romantic atmosphere for our anniversary weekend. The staff even left a complimentary bottle of wine in our room. Unfortunately, the bathroom was severely outdated with a leaky faucet and stained tiles. The clawfoot tub looked nice, but the water pressure was practically non-existent. Lovely ambiance and sweet gestures, but the plumbing definitely needs a modern upgrade.",
  "sentiment": {
    "label": "negative",
    "confidence": 0.4334
  },
  "rating": {
    "prediction": 2,
    "confidence": 0.2966
  },
  "emotion": {
    "dominant": "joy",
    "confidence": 0.7,
    "predictions": [
      {
        "label": "joy",
        "confidence": 0.7
      },
      {
        "label": "neutral",
        "confidence": 0.7
      }
    ]
  },
  "aspects": [
    {
      "aspect": "cleanliness",
      "sentiment": "neutral",
      "confidence": 0.5,
      "evidence": "Unfortunately, the bathroom was severely outdated with a

## Business Traveler Test

In [ ]:
review="The location is unbeatable for corporate meetings. However the Wi-Fi kept dropping every ten minutes and client calls were impossible."
resp=requests.post(f"{BASE_URL}/analyze",json={"review_text":review})
print(json.dumps(resp.json(),indent=2))

## Family Traveler Test

In [ ]:
review="The kids loved the water park and game room. Check-in took over an hour despite our reservation."
resp=requests.post(f"{BASE_URL}/analyze",json={"review_text":review})
print(json.dumps(resp.json(),indent=2))

## Batch API Test

In [ ]:
payload={"reviews":[{"review_text":"Amazing location and service"},{"review_text":"Dirty room and rude staff"}]}
resp=requests.post(f"{BASE_URL}/analyze-batch",json=payload)
print(json.dumps(resp.json(),indent=2))

## Extract Key Fields Into DataFrame

In [ ]:
reviews=["Amazing hotel and great staff","Dirty room and poor maintenance","Family loved the water park but check-in was chaotic"]
rows=[]
for r in reviews:
    d=requests.post(f"{BASE_URL}/analyze",json={"review_text":r}).json()
    rows.append({
      "sentiment":d["sentiment"]["label"],
      "rating":d["rating"]["prediction"],
      "emotion":d["emotion"]["dominant"],
      "quality":d["quality"]["status"]
    })
pd.DataFrame(rows)